# 05 — Basic Decorators

This notebook moves from the **manual decorator pattern** to Python's actual `@decorator` syntax.

The main goal is to understand that `@` is not magic. It is a convenient way to apply a decorator to a function.

Previously:

```python
def decorator(func):

    def wrapper():
        print("Before")
        func()
        print("After")

    return wrapper
```

Now:

```python
@decorator
def greet():
    print("Hello")
```

By the end of this notebook, you should understand exactly how these two forms are related.

## 1. Introduction

The previous notebook built decorators manually.

The basic pattern was:

```text
Function
↓
Function passed as argument
↓
Nested function
↓
Wrapper function
↓
Wrapper returned
↓
Original name reassigned
```

This notebook introduces the cleaner syntax Python provides:

```python
@decorator
def greet():
    ...
```

The key idea is:

> **The `@decorator` syntax is a convenient way to apply a decorator.**

In [ ]:
def decorator(func):

    def wrapper():
        print("Before")
        func()
        print("After")

    return wrapper

---

## 2. What Is a Decorator?

A decorator is a function that:

1. receives another function,
2. adds or changes behavior,
3. returns a function.

Basic pattern:

```python
def decorator(func):

    def wrapper():
        # additional behavior
        func()
        # additional behavior

    return wrapper
```

Decorators are possible because Python functions are **objects**. They can be assigned to variables, passed to other functions, and returned from functions.

In [ ]:
def add_message(func):

    def wrapper():
        print("Starting...")
        func()
        print("Finished.")

    return wrapper

In this example:

- `add_message` receives a function.
- `wrapper` adds behavior before and after that function.
- `add_message` returns `wrapper`.

The original function can therefore be replaced by the returned wrapper.

---

## 3. Review of the Manual Decoration Pattern

Before using `@`, make sure the manual version is clear.

First define the original function:

In [ ]:
def greet():
    print("Hello")

Now define a decorator:

In [ ]:
def add_message(func):

    def wrapper():
        print("Starting...")
        func()
        print("Finished.")

    return wrapper

Apply the decorator manually:

In [ ]:
greet = add_message(greet)

Now call `greet()`:

In [ ]:
greet()

Expected output:

```text
Starting...
Hello
Finished.
```

What happened?

```text
greet
  ↓
add_message(greet)
  ↓
wrapper
  ↓
greet now refers to wrapper
```

This is the manual decorator pattern from the previous notebook.

---

## 4. The `@` Decorator Syntax

Python provides a shorter way to write the same operation.

Instead of:

```python
def greet():
    print("Hello")

greet = add_message(greet)
```

we can write:

In [ ]:
@add_message
def greet():
    print("Hello")

In [ ]:
greet()

Expected output:

```text
Starting...
Hello
Finished.
```

### The most important equivalence

This:

```python
@add_message
def greet():
    print("Hello")
```

is equivalent to:

```python
def greet():
    print("Hello")

greet = add_message(greet)
```

Remember this equivalence. It is the foundation for understanding decorator syntax.

---

## 5. How `@decorator` Works

Consider:

```python
@decorator
def function():
    pass
```

Conceptually, Python applies the decorator like this:

```python
def function():
    pass

function = decorator(function)
```

So the decoration happens when Python **defines the decorated function**.

It does **not** mean:

```python
function = decorator()
```

And it does **not** mean that the decorated function is immediately called.

The actual function call is still:

```python
function()
```

In [ ]:
def decorator(func):

    def wrapper():
        print("Wrapper is running")
        func()

    return wrapper


@decorator
def greet():
    print("Hello")


print("The decorated function has been defined.")
greet()

Notice the sequence:

```text
define greet
    ↓
apply decorator
    ↓
greet refers to the returned wrapper
    ↓
later, greet() is called
```

---

## 6. Creating a Basic Decorator

Build the simplest complete decorator step by step.

The decorator receives the original function:

In [ ]:
def my_decorator(func):

    def wrapper():
        print("Before function")
        func()
        print("After function")

    return wrapper

Apply it using `@`:

In [ ]:
@my_decorator
def greet():
    print("Hello")

Call the decorated function:

In [ ]:
greet()

Expected output:

```text
Before function
Hello
After function
```

The flow is:

```text
my_decorator
     ↓
receives greet
     ↓
creates wrapper
     ↓
returns wrapper
```

The name `greet` then refers to the returned wrapper.

---

## 7. Adding Behavior Before a Function

A decorator can perform an operation before the original function runs.

```python
def before_message(func):

    def wrapper():
        print("Function is about to run")
        func()

    return wrapper
```

In [ ]:
def before_message(func):

    def wrapper():
        print("Function is about to run")
        func()

    return wrapper


@before_message
def greet():
    print("Hello")


greet()

Expected output:

```text
Function is about to run
Hello
```

The original `greet()` function does not contain the extra message. The decorator supplies it.

---

## 8. Adding Behavior After a Function

A decorator can also perform an operation after the original function runs.

In [ ]:
def after_message(func):

    def wrapper():
        func()
        print("Function has completed")

    return wrapper


@after_message
def greet():
    print("Hello")


greet()

Expected output:

```text
Hello
Function has completed
```

---

## 9. Adding Behavior Before and After

This is the classic beginner decorator.

In [ ]:
def log_function(func):

    def wrapper():
        print("Starting function")
        func()
        print("Ending function")

    return wrapper


@log_function
def greet():
    print("Hello")


greet()

Expected output:

```text
Starting function
Hello
Ending function
```

This demonstrates why decorators are useful: the original function does not need to contain the logging code.

---

## 10. Decorators That Return Values

A decorator should normally preserve the behavior of the original function.

First, consider this decorator:

In [ ]:
def decorator(func):

    def wrapper():
        func()

    return wrapper


@decorator
def get_number():
    return 100


result = get_number()
print(result)

The output is:

```text
None
```

Why?

The original function returns `100`, but the wrapper does not return that result.

The wrapper calls:

```python
func()
```

but it does not pass the returned value back to the caller.

Correct the decorator by storing and returning the result:

In [ ]:
def decorator(func):

    def wrapper():
        result = func()
        return result

    return wrapper


@decorator
def get_number():
    return 100


print(get_number())

This can be simplified to:

In [ ]:
def decorator(func):

    def wrapper():
        return func()

    return wrapper


@decorator
def get_number():
    return 100


print(get_number())

The important pattern is:

```python
def wrapper():
    return func()
```

When the original function produces a result, the wrapper should return that result if the decorator is intended to preserve the original behavior.

---

## 11. Decorators with Function Arguments

Now consider a function that accepts an argument:

```python
@decorator
def greet(name):
    print("Hello", name)
```

The wrapper must also accept the argument.

In [ ]:
def decorator(func):

    def wrapper(name):
        print("Before")
        func(name)
        print("After")

    return wrapper


@decorator
def greet(name):
    print("Hello", name)


greet("Komal")

Expected output:

```text
Before
Hello Komal
After
```

Now consider a function with two arguments.

In [ ]:
def decorator(func):

    def wrapper(a, b):
        print("Before")
        result = func(a, b)
        print("After")
        return result

    return wrapper


@decorator
def add(a, b):
    return a + b


print(add(10, 20))

Expected output:

```text
Before
After
30
```

### Important limitation

At this stage, the wrapper has to know the original function's arguments.

For example, a wrapper written as:

```python
def wrapper(a, b):
    ...
```

is not suitable for a function that expects three arguments or different keyword arguments.

Later in the decorators sequence, `*args` and `**kwargs` will be used to build more flexible wrappers.

For this notebook, keep the focus on the basic decorator mechanism.

---

## 12. Multiple Functions Using the Same Decorator

One decorator can be reused with multiple functions.

In [ ]:
def log_function(func):

    def wrapper():
        print("Running:", func.__name__)
        func()

    return wrapper


@log_function
def greet():
    print("Hello")


@log_function
def goodbye():
    print("Goodbye")


greet()
goodbye()

Expected output:

```text
Running: greet
Hello
Running: goodbye
Goodbye
```

The same decorator function was applied to both `greet` and `goodbye`.

This demonstrates the reusable nature of decorators.

---

## 13. Stacking Multiple Decorators

Python allows multiple decorators on the same function.

First, define two decorators:

In [ ]:
def decorator_one(func):

    def wrapper():
        print("Decorator One")
        func()

    return wrapper


def decorator_two(func):

    def wrapper():
        print("Decorator Two")
        func()

    return wrapper

Apply both decorators:

In [ ]:
@decorator_one
@decorator_two
def greet():
    print("Hello")


greet()

Expected output:

```text
Decorator One
Decorator Two
Hello
```

The stacked form:

```python
@decorator_one
@decorator_two
def greet():
    print("Hello")
```

is conceptually equivalent to:

```python
greet = decorator_one(decorator_two(greet))
```

This introduces **decorator stacking**.

---

## 14. Order of Decorator Execution

The order is important.

Given:

```python
@decorator_one
@decorator_two
def greet():
    print("Hello")
```

### Decoration happens from the bottom upward

Conceptually:

```text
decorator_two(greet)
        ↓
decorator_one(result)
```

So the final assignment is:

```python
greet = decorator_one(decorator_two(greet))
```

### Calling happens through the outermost wrapper first

The resulting call flows like this:

```text
decorator_one
    ↓
decorator_two
    ↓
greet
```

Visualize the structure as:

```text
@decorator_one
@decorator_two
def greet():
    ...
```

becoming:

```text
decorator_one(
    decorator_two(
        greet
    )
)
```

A useful rule is:

> **Decorators are applied from the bottom upward, but the resulting wrappers execute from the outside inward when the function is called.**

In [ ]:
def decorator_one(func):

    def wrapper():
        print("One - before")
        func()
        print("One - after")

    return wrapper


def decorator_two(func):

    def wrapper():
        print("Two - before")
        func()
        print("Two - after")

    return wrapper


@decorator_one
@decorator_two
def greet():
    print("Hello")


greet()

Expected output:

```text
One - before
Two - before
Hello
Two - after
One - after
```

This makes the nesting visible.

---

## 15. Practical Decorator Examples

### Example 1 — Logging

A logging-style decorator can show when a function starts and finishes.

In [ ]:
def log_call(func):

    def wrapper():
        print("Calling:", func.__name__)
        result = func()
        print("Completed:", func.__name__)
        return result

    return wrapper


@log_call
def greet():
    print("Hello")


greet()

---

### Example 2 — Authentication Concept

This example demonstrates the idea of checking access before allowing a function to run.

It is only a simple learning example, not a complete authentication system.

In [ ]:
def require_login(func):

    def wrapper():
        logged_in = True

        if logged_in:
            return func()

        print("Access denied")


@require_login
def dashboard():
    print("Welcome to dashboard")


dashboard()

---

### Example 3 — Simple Validation

A decorator can place validation-related behavior before a function.

In [ ]:
def validate_user(func):

    def wrapper():
        print("Validating user...")
        return func()

    return wrapper


@validate_user
def create_account():
    print("Account created")


create_account()

---

### Example 4 — Before/After Execution

A decorator can surround a calculation while preserving its result.

In [ ]:
def show_execution(func):

    def wrapper():
        print("START")
        result = func()
        print("END")
        return result


@show_execution
def calculate():
    return 10 + 20


print(calculate())

Expected output:

```text
START
END
30
```

---

## 16. Common Mistakes

### Mistake 1 — Forgetting to return the wrapper

Wrong:

```python
def decorator(func):

    def wrapper():
        func()
```

Correct:

```python
def decorator(func):

    def wrapper():
        func()

    return wrapper
```

The decorator itself needs to return the replacement function.

---

### Mistake 2 — Calling the function while decorating

Wrong:

```python
def decorator(func):
    func()

    def wrapper():
        print("Wrapper")

    return wrapper
```

Here `func()` executes while the decoration is happening.

A decorator normally **receives** the function and returns a wrapper. The wrapper calls the original function later.

---

### Mistake 3 — Forgetting the result

Wrong:

```python
def decorator(func):

    def wrapper():
        func()

    return wrapper
```

If `func()` returns a value and the wrapper does not return it, the decorated call receives `None`.

Correct:

```python
def decorator(func):

    def wrapper():
        return func()

    return wrapper
```

---

### Mistake 4 — Confusing these two forms

```python
@decorator
def greet():
    pass
```

and:

```python
@decorator()
def greet():
    pass
```

For this notebook, focus on the first form.

The second form represents a **decorator factory / decorator with arguments**, which belongs later in the decorators sequence.

---

### Mistake 5 — Thinking `@` calls the decorated function

It does not.

```python
@decorator
def greet():
    print("Hello")
```

The decorator is applied when the function is defined.

The actual function call is still:

```python
greet()
```

---

## 17. Manual Decoration vs `@` Syntax

This comparison ties this notebook directly to the previous one.

### Manual decoration

```python
def greet():
    print("Hello")

greet = decorator(greet)
```

### `@` syntax

```python
@decorator
def greet():
    print("Hello")
```

### They are equivalent

```python
@decorator
def greet():
    print("Hello")
```

means:

```python
def greet():
    print("Hello")

greet = decorator(greet)
```

The `@` form is simply the cleaner syntax for applying the decorator.

In [ ]:
# Manual form

def manual_decorator(func):

    def wrapper():
        print("Before")
        result = func()
        print("After")
        return result

    return wrapper


def manual_greet():
    print("Hello")


manual_greet = manual_decorator(manual_greet)


manual_greet()

In [ ]:
# @ syntax

def syntax_decorator(func):

    def wrapper():
        print("Before")
        result = func()
        print("After")
        return result

    return wrapper


@syntax_decorator
def syntax_greet():
    print("Hello")


syntax_greet()

---

## 18. Summary

### What is a decorator?

```text
A function that receives another function
and returns a modified or replacement function.
```

### Basic structure

```python
def decorator(func):

    def wrapper():
        # extra behavior
        result = func()
        # extra behavior
        return result

    return wrapper
```

### Applying it

```python
@decorator
def greet():
    print("Hello")
```

### Equivalent manual form

```python
def greet():
    print("Hello")

greet = decorator(greet)
```

### Multiple decorators

```python
@decorator_one
@decorator_two
def greet():
    ...
```

means:

```python
greet = decorator_one(decorator_two(greet))
```

### Final mental model

```text
Function
    ↓
Decorator receives function
    ↓
Decorator creates wrapper
    ↓
Decorator returns wrapper
    ↓
@ syntax assigns the returned wrapper
    ↓
Calling the function runs the wrapper
    ↓
Wrapper can call the original function
```

The next topics in the decorators sequence will build on this foundation rather than changing the basic mechanism.

### Boundary with the Next Notebooks

Keep this notebook focused on **basic decorator mechanics**.

The following topics are intentionally deferred:

- `*args` and `**kwargs` for universal wrappers
- preserving function metadata with `functools.wraps`
- decorators that themselves accept arguments
- class decorators
- advanced practical decorator patterns

Those topics belong later in the decorators sequence.